# 🤖 Spot RL Agent — Interactive Demo

Notebook này demo cách DQN agent hành động trong môi trường Spot Instance:
1. **Episode Replay** — Tua từng step bằng slider, xem agent quyết định gì
2. **Policy Heatmap** — Agent chọn action gì theo (price × workload)
3. **So sánh Agents** — DQN vs Always-Spot vs Always-OnDemand

In [6]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch
import warnings
warnings.filterwarnings('ignore')

from envs.spot_env import SpotInstanceEnv
from agents.dqn_agent import DQNAgent

print('✅ Imports OK')

✅ Imports OK


## ⚙️ Cấu hình — Chọn model và dataset

In [7]:
# ── Chọn model muốn demo ──────────────────────────────────────────────
AVAILABLE_MODELS = {
    'quick_test_v2 (best cost)' : '../results/models/quick_test_v2_best.pth',
    'quick_test_v3 (best SLA)'  : '../results/models/quick_test_v3_best.pth',
    'quick_test'                : '../results/models/quick_test_best.pth',
    'dqn_stable_v5 (long train)': '../results/dqn_stable_v5/20260324_130359/models/best_model.pth',
    'dqn_volatile_v5'           : '../results/models/dqn_volatile_v5_best.pth',
    'dqn_spike_v5'              : '../results/models/dqn_spike_v5_best.pth',
}

AVAILABLE_DATA = {
    'stable'  : '../data/processed/price_features_stable.pkl',
    'volatile': '../data/processed/price_features_volatile.pkl',
    'spike'   : '../data/processed/price_features_spike.pkl',
    'default' : '../data/processed/price_features.pkl',
}

# ENV config (khớp với dqn_quick_test.yaml)
ENV_CONFIG = dict(
    max_steps=200,
    sla_threshold=0.95,
    spot_capacity=10,
    ondemand_capacity=5,
    workload_config=dict(base_arrival_rate=2.0, peak_multiplier=3.0, avg_job_duration=10),
    cost_config=dict(ondemand_price=0.096, sla_penalty=2.0, migration_cost=0.3),
)

# Action colors & labels
ACTION_NAMES  = ['REQ_SPOT', 'REQ_OD', 'TERM_SPOT', 'TERM_OD', 'MIG→OD', 'MIG→SPOT', 'IDLE']
ACTION_COLORS = ['#2ecc71', '#e74c3c', '#f39c12', '#c0392b', '#9b59b6', '#3498db', '#95a5a6']
SLA_THRESHOLD = 0.95

print('✅ Config OK')

✅ Config OK


## 📦 Load model & chạy episode

In [8]:
def load_agent(model_path: str) -> DQNAgent:
    agent = DQNAgent(state_dim=15, action_dim=7)
    agent.load(model_path)
    agent.epsilon = 0.0  # pure exploitation
    return agent


def run_episode(model_key: str, data_key: str, seed: int = 42) -> pd.DataFrame:
    """Chạy 1 episode với agent đã train, trả về history DataFrame."""
    model_path = AVAILABLE_MODELS[model_key]
    data_path  = AVAILABLE_DATA[data_key]

    env = SpotInstanceEnv(data_path=data_path, **ENV_CONFIG)
    agent = load_agent(model_path)

    obs, _ = env.reset(seed=seed)
    done = False
    while not done:
        action = agent.select_action(obs, training=False)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    df = env.get_episode_history()
    print(f'✅ Episode done | Steps: {len(df)} | '
          f'Total cost: ${df["total_cost"].iloc[-1]:.2f} | '
          f'Final SLA: {df["sla"].iloc[-1]:.1%}')
    return df


# ── Widgets chọn model / dataset ──────────────────────────────────────
w_model = widgets.Dropdown(
    options=list(AVAILABLE_MODELS.keys()),
    value='quick_test_v2 (best cost)',
    description='Model:',
    layout=widgets.Layout(width='320px'),
)
w_data = widgets.Dropdown(
    options=list(AVAILABLE_DATA.keys()),
    value='stable',
    description='Dataset:',
    layout=widgets.Layout(width='320px'),
)
w_seed = widgets.IntSlider(value=42, min=0, max=200, description='Seed:', layout=widgets.Layout(width='320px'))
w_run  = widgets.Button(description='▶ Run Episode', button_style='success',
                        layout=widgets.Layout(width='160px'))
out_run = widgets.Output()

episode_data = {}  # store result

def on_run(_):
    with out_run:
        clear_output()
        print(f'Running {w_model.value} on {w_data.value} data (seed={w_seed.value})...')
        df = run_episode(w_model.value, w_data.value, seed=w_seed.value)
        episode_data['df']    = df
        episode_data['model'] = w_model.value
        episode_data['data']  = w_data.value

w_run.on_click(on_run)
display(widgets.HBox([w_model, w_data, w_seed]), w_run, out_run)

Button(button_style='success', description='▶ Run Episode', layout=Layout(width='160px'), style=ButtonStyle())

Output()

---
## 1️⃣ Episode Replay — Tua từng Step bằng Slider

In [9]:
def plot_step_detail(df, step):
    # Need >=2 points for fill_between/stackplot; pad if step=0
    end = max(step + 1, 2)
    hist = df.iloc[:end].copy()
    row  = df.iloc[step]
    xs   = hist['step'].values.astype(float)

    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    action_idx   = int(row['action'])
    action_label = ACTION_NAMES[action_idx]
    action_color = ACTION_COLORS[action_idx]

    fig.suptitle(
        'Step {}/{} | Action: {} | Price: ${:.4f}/hr | SLA: {:.1%} | Cost: ${:.2f}'.format(
            step+1, len(df), action_label,
            row['spot_price'], row['sla'], row['total_cost']
        ),
        fontsize=13, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=action_color, alpha=0.25)
    )

    # Row 0: price timeline
    ax0 = fig.add_subplot(gs[0, :])
    prices_arr = hist['spot_price'].values.astype(float)
    ax0.plot(xs, prices_arr, color='#2c3e50', lw=1.5)
    ax0.fill_between(xs, prices_arr, alpha=0.1, color='#2c3e50')
    for _, r in hist.iterrows():
        ax0.axvline(float(r['step']), color=ACTION_COLORS[int(r['action'])], alpha=0.25, lw=0.8)
    ax0.axvline(float(row['step']), color='red', lw=2, linestyle='--')
    ax0.set_ylabel('Spot Price ($/hr)', fontsize=9)
    ax0.set_title('Spot Price Timeline (vach mau = action)', fontsize=10)
    patches = [mpatches.Patch(color=ACTION_COLORS[i], label=ACTION_NAMES[i]) for i in range(7)]
    ax0.legend(handles=patches, ncol=7, loc='upper right', fontsize=7)

    # Row 1: instances
    ax1 = fig.add_subplot(gs[1, 0])
    spot_arr = hist['spot_instances'].values.astype(float)
    od_arr   = hist['ondemand_instances'].values.astype(float)
    ax1.stackplot(xs, spot_arr, od_arr,
                  labels=['Spot', 'On-demand'],
                  colors=['#3498db', '#e74c3c'], alpha=0.75)
    ax1.axvline(float(row['step']), color='red', lw=1.5, linestyle='--')
    ax1.set_title('Instances', fontsize=10)
    ax1.set_ylabel('Count', fontsize=9)
    ax1.legend(fontsize=8, loc='upper left')

    # Row 1: SLA
    ax2 = fig.add_subplot(gs[1, 1])
    sla_arr = hist['sla'].values.astype(float) * 100
    ax2.plot(xs, sla_arr, color='#27ae60', lw=1.5)
    ax2.fill_between(xs, sla_arr, alpha=0.15, color='#27ae60')
    ax2.axhline(SLA_THRESHOLD * 100, color='red', linestyle='--', lw=1.2, label='Target 95%')
    ax2.axvline(float(row['step']), color='red', lw=1.5, linestyle='--')
    ax2.set_ylim(0, 105)
    ax2.set_title('SLA Compliance', fontsize=10)
    ax2.set_ylabel('%', fontsize=9)
    ax2.legend(fontsize=8)

    # Row 1: cost
    ax3 = fig.add_subplot(gs[1, 2])
    cost_arr = hist['total_cost'].values.astype(float)
    ax3.plot(xs, cost_arr, color='#e67e22', lw=1.5)
    ax3.fill_between(xs, cost_arr, alpha=0.15, color='#e67e22')
    ax3.axvline(float(row['step']), color='red', lw=1.5, linestyle='--')
    ax3.set_title('Cumulative Cost ($)', fontsize=10)
    ax3.set_ylabel('$', fontsize=9)

    # Row 2: jobs
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.plot(xs, hist['pending_jobs'].values.astype(float), color='#e74c3c', lw=1.2, label='Pending')
    if 'running_jobs' in hist.columns:
        ax4.plot(xs, hist['running_jobs'].values.astype(float), color='#2ecc71', lw=1.2, label='Running')
    ax4.axvline(float(row['step']), color='red', lw=1.5, linestyle='--')
    ax4.set_title('Job Queue', fontsize=10)
    ax4.set_ylabel('Jobs', fontsize=9)
    ax4.legend(fontsize=8)

    # Row 2: reward
    ax5 = fig.add_subplot(gs[2, 1])
    actual_hist = df.iloc[:step + 1]
    actual_xs   = actual_hist['step'].values.astype(float)
    rewards = actual_hist['reward'].values.astype(float)
    rcolors = ['#27ae60' if v >= 0 else '#e74c3c' for v in rewards]
    bar_w = max(0.5, 150.0 / max(len(actual_xs), 1))
    ax5.bar(actual_xs, rewards, color=rcolors, alpha=0.7, width=bar_w)
    ax5.axhline(0, color='black', lw=0.8)
    ax5.axvline(float(row['step']), color='red', lw=1.5, linestyle='--')
    ax5.set_title('Reward per Step', fontsize=10)
    ax5.set_ylabel('Reward', fontsize=9)

    # Row 2: action dist
    ax6 = fig.add_subplot(gs[2, 2])
    action_counts = actual_hist['action'].value_counts().reindex(range(7), fill_value=0)
    ax6.barh(ACTION_NAMES, action_counts.values, color=ACTION_COLORS, alpha=0.8)
    ax6.set_title('Action Distribution (so far)', fontsize=10)
    ax6.set_xlabel('Count', fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
    plt.close(fig)

# ── Interactive slider ────────────────────────────────────────────────
out_replay = widgets.Output()

def show_replay(step):
    if 'df' not in episode_data:
        print('⚠️ Chưa có dữ liệu — hãy nhấn "▶ Run Episode" ở cell trên trước!')
        return
    with out_replay:
        clear_output(wait=True)
        plot_step_detail(episode_data['df'], step)

w_step = widgets.IntSlider(
    value=0, min=0, max=199, step=1,
    description='Step:',
    continuous_update=False,
    layout=widgets.Layout(width='80%'),
    style={'description_width': '50px'}
)

def on_step_change(change):
    if 'df' in episode_data:
        max_step = len(episode_data['df']) - 1
        w_step.max = max_step
    show_replay(change['new'])

w_step.observe(on_step_change, names='value')

# Play button
w_play = widgets.Play(value=0, min=0, max=199, step=5, interval=400, description='Auto-play')
widgets.jslink((w_play, 'value'), (w_step, 'value'))

display(
    widgets.HTML('<h3>📽️ Episode Replay</h3><p>Chạy episode ở trên trước, rồi kéo slider hoặc nhấn play.</p>'),
    widgets.HBox([w_play, w_step]),
    out_replay
)

HTML(value='<h3>📽️ Episode Replay</h3><p>Chạy episode ở trên trước, rồi kéo slider hoặc nhấn play.</p>')

Output()

---
## 2️⃣ Policy Heatmap — Agent chọn action gì theo (Price × Workload)

In [ ]:
def build_policy_heatmap(model_key: str, data_key: str,
                          price_res: int = 30, workload_res: int = 30):
    """Freeze model, quét state space, vẽ heatmap action."""
    agent = load_agent(AVAILABLE_MODELS[model_key])

    prices    = np.linspace(0.01, 0.20, price_res)
    workloads = np.linspace(0, 50, workload_res)

    # Các giá trị state còn lại giữ ở mức trung bình
    base_state = np.array([
        0.07,  # price → override
        0.07,  # price_ma_6h
        0.07,  # price_ma_24h
        0.01,  # volatility
        0.2,   # interruption_prob
        5.0,   # spot_instances
        2.0,   # ondemand_instances
        7.0,   # total_capacity
        5.0,   # pending_jobs → override
        5.0,   # running_jobs
        10.0,  # workload_forecast
        2.0,   # avg_wait
        12.0,  # hour_of_day
        2.0,   # day_of_week
        0.5,   # progress
    ], dtype=np.float32)

    # Normalization bounds (từ SpotInstanceEnv)
    obs_max = np.array([0.20, 0.20, 0.20, 0.05, 1.0,
                        10.0, 5.0, 15.0, 100.0, 15.0,
                        100.0, 100.0, 23.0, 6.0, 1.0], dtype=np.float32)

    action_grid  = np.zeros((workload_res, price_res), dtype=int)
    qvalue_grid  = np.zeros((workload_res, price_res))

    for i, wl in enumerate(workloads):
        for j, pr in enumerate(prices):
            s = base_state.copy()
            s[0] = pr   # current_price
            s[1] = pr   # price_ma_6h
            s[8] = wl   # pending_jobs
            # Normalize
            s_norm = np.clip(s / np.where(obs_max == 0, 1.0, obs_max), 0.0, 1.0)
            with torch.no_grad():
                t = torch.FloatTensor(s_norm).unsqueeze(0).to(agent.device)
                q = agent.q_network(t)
                action_grid[i, j] = q.argmax().item()
                qvalue_grid[i, j] = q.max().item()

    # ── Plot ──────────────────────────────────────────────────────────
    from matplotlib.colors import ListedColormap
    cmap7 = ListedColormap(ACTION_COLORS)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'Policy Heatmap — {model_key}', fontsize=13, fontweight='bold')

    # Action heatmap
    im0 = axes[0].imshow(
        action_grid, aspect='auto', origin='lower',
        extent=[prices[0], prices[-1], workloads[0], workloads[-1]],
        cmap=cmap7, vmin=0, vmax=6
    )
    axes[0].set_xlabel('Spot Price ($/hr)', fontsize=11)
    axes[0].set_ylabel('Pending Jobs (workload)', fontsize=11)
    axes[0].set_title('Action Agent Chọn', fontsize=11)
    cbar0 = plt.colorbar(im0, ax=axes[0], ticks=range(7))
    cbar0.set_ticklabels(ACTION_NAMES)

    # Max Q-value heatmap
    im1 = axes[1].imshow(
        qvalue_grid, aspect='auto', origin='lower',
        extent=[prices[0], prices[-1], workloads[0], workloads[-1]],
        cmap='RdYlGn'
    )
    axes[1].set_xlabel('Spot Price ($/hr)', fontsize=11)
    axes[1].set_ylabel('Pending Jobs (workload)', fontsize=11)
    axes[1].set_title('Max Q-value (confidence)', fontsize=11)
    plt.colorbar(im1, ax=axes[1])

    plt.tight_layout()
    plt.show()


out_heatmap = widgets.Output()
w_hm_model  = widgets.Dropdown(
    options=list(AVAILABLE_MODELS.keys()),
    value='quick_test_v2 (best cost)',
    description='Model:',
    layout=widgets.Layout(width='320px'),
)
w_hm_btn = widgets.Button(description='🗺️ Generate Heatmap', button_style='info',
                           layout=widgets.Layout(width='200px'))

def on_heatmap(_):
    with out_heatmap:
        clear_output(wait=True)
        print('Generating...')
        build_policy_heatmap(w_hm_model.value, 'stable')

w_hm_btn.on_click(on_heatmap)
display(
    widgets.HTML('<h3>🗺️ Policy Heatmap</h3>'),
    widgets.HBox([w_hm_model, w_hm_btn]),
    out_heatmap
)

HTML(value='<h3>🗺️ Policy Heatmap</h3>')

Output()

---
## 3️⃣ So sánh Agents — DQN vs Always-Spot vs Always-OnDemand

In [11]:
def run_baseline_episode(env_config: dict, data_path: str,
                          strategy: str, seed: int = 42) -> pd.DataFrame:
    """Chạy episode với baseline policy."""
    env = SpotInstanceEnv(data_path=data_path, **env_config)
    obs, _ = env.reset(seed=seed)
    done = False
    step = 0
    while not done:
        if strategy == 'always_spot':
            # Luôn request spot nếu chưa đầy, không làm gì nếu đầy
            if env.num_spot_instances < env.spot_capacity:
                action = SpotInstanceEnv.ACTION_REQUEST_SPOT
            else:
                action = SpotInstanceEnv.ACTION_DO_NOTHING
        elif strategy == 'always_ondemand':
            if env.num_ondemand_instances < env.ondemand_capacity:
                action = SpotInstanceEnv.ACTION_REQUEST_ONDEMAND
            else:
                action = SpotInstanceEnv.ACTION_DO_NOTHING
        elif strategy == 'mixed':
            # 70% spot, 30% on-demand
            total = env.num_spot_instances + env.num_ondemand_instances
            if total == 0:
                action = SpotInstanceEnv.ACTION_REQUEST_SPOT
            elif env.num_spot_instances / max(total, 1) < 0.7 and env.num_spot_instances < env.spot_capacity:
                action = SpotInstanceEnv.ACTION_REQUEST_SPOT
            elif env.num_ondemand_instances < env.ondemand_capacity:
                action = SpotInstanceEnv.ACTION_REQUEST_ONDEMAND
            else:
                action = SpotInstanceEnv.ACTION_DO_NOTHING
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        step += 1
    return env.get_episode_history()


def compare_agents(model_key: str, data_key: str, seed: int = 42):
    data_path = AVAILABLE_DATA[data_key]

    print('Running DQN agent...')
    df_dqn = run_episode(model_key, data_key, seed=seed)
    print('Running Always-Spot baseline...')
    df_spot = run_baseline_episode(ENV_CONFIG, data_path, 'always_spot', seed=seed)
    print('Running Always-OnDemand baseline...')
    df_od = run_baseline_episode(ENV_CONFIG, data_path, 'always_ondemand', seed=seed)
    print('Running Mixed (70/30) baseline...')
    df_mix = run_baseline_episode(ENV_CONFIG, data_path, 'mixed', seed=seed)

    agents = {
        f'DQN ({model_key})': (df_dqn, '#e74c3c', '-'),
        'Always Spot'       : (df_spot, '#3498db', '--'),
        'Always OnDemand'   : (df_od, '#e67e22', '--'),
        'Mixed 70/30'       : (df_mix, '#9b59b6', ':'),
    }

    fig, axes = plt.subplots(2, 2, figsize=(15, 9))
    fig.suptitle(f'Agent Comparison — {data_key} market (seed={seed})',
                 fontsize=14, fontweight='bold')

    metrics = [
        ('total_cost',  axes[0, 0], 'Cumulative Cost ($)',     '$ (↓ better)'),
        ('sla',         axes[0, 1], 'SLA Compliance',          '% (↑ ≥95% target)'),
        ('spot_instances', axes[1, 0], 'Spot Instances Used',  'Count'),
        ('reward',      axes[1, 1], 'Reward per Step',         'Reward (↑ better)'),
    ]

    for col, ax, title, ylabel in metrics:
        for label, (df, color, ls) in agents.items():
            steps = df['step'].values
            vals  = df[col].values
            if col == 'sla':
                vals = vals * 100
            ax.plot(steps, vals, color=color, linestyle=ls, lw=1.8, label=label)
        if col == 'sla':
            ax.axhline(95, color='red', linestyle='--', lw=1, label='Target 95%')
            ax.set_ylim(0, 105)
        ax.set_title(title, fontsize=11)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_xlabel('Step', fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary table
    print('\n── Summary (last step) ─────────────────────────────────────────')
    rows = []
    for label, (df, _, _) in agents.items():
        last = df.iloc[-1]
        spot_ratio = df['spot_instances'].mean() / max(
            (df['spot_instances'] + df['ondemand_instances']).mean(), 1) * 100
        rows.append({
            'Agent'      : label,
            'Total Cost' : f"${last['total_cost']:.2f}",
            'Final SLA'  : f"{last['sla']:.1%}",
            'Avg Spot %' : f"{spot_ratio:.1f}%",
            'Total Reward': f"{df['reward'].sum():.1f}",
        })
    display(pd.DataFrame(rows).set_index('Agent'))


out_compare  = widgets.Output()
w_cmp_model  = widgets.Dropdown(
    options=list(AVAILABLE_MODELS.keys()),
    value='quick_test_v2 (best cost)',
    description='Model:',
    layout=widgets.Layout(width='320px'),
)
w_cmp_data  = widgets.Dropdown(
    options=list(AVAILABLE_DATA.keys()),
    value='stable',
    description='Dataset:',
    layout=widgets.Layout(width='280px'),
)
w_cmp_seed  = widgets.IntSlider(value=42, min=0, max=200,
                                 description='Seed:', layout=widgets.Layout(width='280px'))
w_cmp_btn   = widgets.Button(description='⚡ Compare Agents', button_style='warning',
                              layout=widgets.Layout(width='200px'))

def on_compare(_):
    with out_compare:
        clear_output(wait=True)
        compare_agents(w_cmp_model.value, w_cmp_data.value, seed=w_cmp_seed.value)

w_cmp_btn.on_click(on_compare)
display(
    widgets.HTML('<h3>⚡ Agent Comparison</h3>'),
    widgets.HBox([w_cmp_model, w_cmp_data, w_cmp_seed]),
    w_cmp_btn,
    out_compare
)

HTML(value='<h3>⚡ Agent Comparison</h3>')

Button(button_style='warning', description='⚡ Compare Agents', layout=Layout(width='200px'), style=ButtonStyle…

Output()